## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 12 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,2157809066,4378876,2026-07-15,2026-07-15,1,NI,3836.000000000,BLEND3_3,4378876,"{""pessoas"":[{""nome"":""CALIANDRA AVELLO SCHIRMER...",...,33,1778785248777,"{""valor_aluguel"":2500,""matchmaking_on"":false,""...",2026-07-16 08:00:43+00:00,1784188841029067190,02157809066,E,2026-07-16 08:01:28.715000+00:00,2026-07-15 17:02:30+00:00,REPROVAR
1,12886286923,4388611,2026-07-17,2026-07-17,1,C,1918.000000000,BLEND3_3,4388611,"{""pessoas"":[{""nome"":""JOAO NEGREIROS TAIRA"",""do...",...,32,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-18 08:00:27+00:00,1784361623840494393,12886286923,B,2026-07-18 08:02:32.598000+00:00,2026-07-17 15:49:55+00:00,APROVAR
2,37519245829,4390386,2026-07-18,2026-07-18,1,D,2534.500000000,BLEND3_3,4390386,"{""pessoas"":[{""nome"":""DAVID MAURO LENSO"",""docum...",...,33,1778785248777,"{""valor_aluguel"":1200,""matchmaking_on"":false,""...",2026-07-18 18:00:38+00:00,1784397636498950335,37519245829,E,2026-07-18 18:03:11.184000+00:00,2026-07-18 10:50:21+00:00,REPROVAR
3,9337705966,4078949,2026-04-30,2026-04-30,1,C,1370.000000000,BLEND3_3,4078949,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,32,1778785248777,None,2026-05-20 08:01:26+00:00,1779264084763474336,09337705966,B,2026-05-20 08:09:26.185000+00:00,2026-04-30 08:58:00+00:00,APROVAR
4,4893443801,4079197,2026-04-30,2026-04-30,1,B,2603.000000000,BLEND3_3,4079197,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,34,1778785248777,None,2026-05-20 08:01:28+00:00,1779264086333491387,04893443801,C,2026-05-20 08:09:25.954000+00:00,2026-04-30 09:28:15+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328372,19227765727,4409143,2026-07-22,2026-07-22,1,NI,1644.000000000,BLEND3_3,4409143,"{""pessoas"":[{""nome"":""BRENDA COSTA PAIVA"",""docu...",...,34,1778785248777,"{""valor_aluguel"":950,""matchmaking_on"":false,""p...",2026-07-23 08:00:28+00:00,1784793626011385373,19227765727,C,2026-07-23 08:07:31.804000+00:00,2026-07-22 17:40:20+00:00,APROVAR
328373,388247959,4407648,2026-07-22,2026-07-22,1,D,6987.000000000,BLEND_4,4407648,"{""pessoas"":[{""nome"":""VALDICLEA APARECIDA MARIA...",...,33,1778785248777,"{""principalDocument"":""00388247959"",""imobiliari...",2026-07-23 08:00:28+00:00,1784793623336563377,00388247959,E,2026-07-23 08:07:29.455000+00:00,2026-07-22 15:30:57+00:00,REPROVAR
328374,5587133192,4409322,2026-07-22,2026-07-22,1,NI,5343.000000000,BLEND3_3,4409322,"{""pessoas"":[{""nome"":""RHONIELLY CRISLAINE BRAGA...",...,37,1778785248777,"{""valor_aluguel"":2200,""matchmaking_on"":false,""...",2026-07-23 08:00:28+00:00,1784793626342033737,05587133192,E,2026-07-23 08:07:32.117000+00:00,2026-07-22 18:03:56+00:00,DERIVAR
328375,5880713156,4409072,2026-07-22,2026-07-22,1,E,0E-9,BLEND3_3,4409072,"{""pessoas"":[{""nome"":"""",""documento"":""0588071315...",...,33,1778785248777,"{""valor_aluguel"":1800,""matchmaking_on"":false,""...",2026-07-23 08:00:28+00:00,1784793625869577591,05880713156,E,2026-07-23 08:07:31.703000+00:00,2026-07-22 17:33:11+00:00,REPROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

## Quebrar JSON - Retornado

In [7]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [8]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [9]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [10]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [11]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [12]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [13]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
25197,0.000000,0.017544
25248,0.000000,0.000000
25256,0.000000,0.113043
25263,0.000000,0.077273
25415,0.000000,0.117021
...,...,...
321194,NaN,NaN
321203,0.000000,0.000000
321291,NaN,NaN
321356,0.133065,0.123300


## Escoragem Blend4

In [14]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [15]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [16]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [17]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,2157809066,4378876,2026-07-15,2026-07-15,1,NI,BLEND3_3,6103963,33,02157809066,...,NaN,NaN,NaN,2.0,911.0,"CRED CARTAO,DUPLICATA",BLEND4,NaN,NaN,3836.0
1,12886286923,4388611,2026-07-17,2026-07-17,1,C,BLEND3_3,6115969,32,12886286923,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1918.0
2,37519245829,4390386,2026-07-18,2026-07-18,1,D,BLEND3_3,6118204,33,37519245829,...,NaN,NaN,NaN,2.0,440.0,"DUPLICATA,FINANCIAMENT",BLEND4,NaN,NaN,2534.5
3,9337705966,4078949,2026-04-30,2026-04-30,1,C,BLEND3_3,5731599,32,09337705966,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1370.0
4,4893443801,4079197,2026-04-30,2026-04-30,1,B,BLEND3_3,5731904,34,04893443801,...,NaN,NaN,NaN,1.0,70.0,FAT AGUA,BLEND4,NaN,NaN,2603.0


In [18]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    327265
E_BVS       1111
dtype: int64

In [19]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [20]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [21]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [22]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [23]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,2157809066,4378876,2026-07-15,2026-07-15,1,NI,BLEND3_3,6103963,33,02157809066,...,0.000000,0.05,3.0,0.489880,0.000000,0.395682,0.558456,442.0,C,E
1,12886286923,4388611,2026-07-17,2026-07-17,1,C,BLEND3_3,6115969,32,12886286923,...,0.000000,-0.65,0.0,0.892963,-0.000439,-0.554629,0.547764,452.0,B,A
2,37519245829,4390386,2026-07-18,2026-07-18,1,D,BLEND3_3,6118204,33,37519245829,...,0.000000,0.20,3.0,0.096134,0.000000,0.705563,0.560014,440.0,C,E
3,9337705966,4078949,2026-04-30,2026-04-30,1,C,BLEND3_3,5731599,32,09337705966,...,0.000000,-0.75,0.0,0.000000,-0.059967,-0.551433,0.526606,473.0,B,A
4,4893443801,4079197,2026-04-30,2026-04-30,1,B,BLEND3_3,5731904,34,04893443801,...,0.000000,1.90,1.0,0.000000,0.000000,-0.486094,0.487289,513.0,A,C
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
328371,19227765727,4409143,2026-07-22,2026-07-22,1,NI,BLEND3_3,6141409,34,19227765727,...,0.000000,-0.35,0.0,0.326727,0.000000,-0.618537,0.497308,503.0,A,B
328372,388247959,4407648,2026-07-22,2026-07-22,1,D,BLEND_4,6139529,33,00388247959,...,-0.542274,0.85,6.0,-0.364838,0.030690,0.150808,0.730316,270.0,E,E
328373,5587133192,4409322,2026-07-22,2026-07-22,1,NI,BLEND3_3,6141630,37,05587133192,...,0.000000,-0.40,0.0,-0.040182,0.000000,1.326574,0.564058,436.0,C,D
328374,5880713156,4409072,2026-07-22,2026-07-22,1,E,BLEND3_3,6141324,33,05880713156,...,0.000000,0.00,0.0,0.000000,0.073942,1.326574,0.592344,408.0,C,None


In [24]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            41
BLEND3_3                278043
BLEND_4                  10376
BLEND_REGRESSAO_2026     20654
BVS_CUSTOM                4201
BVS_CUSTOM_V2              162
HFT1                       126
HVA3                     12813
HVA4                      1960
dtype: int64

In [25]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [26]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    327265
E_BVS       1111
dtype: int64

In [27]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            41
BLEND3_3                278043
BLEND_4                  10376
BLEND_REGRESSAO_2026     20654
BVS_CUSTOM                4201
BVS_CUSTOM_V2              162
HFT1                       126
HVA3                     12813
HVA4                      1960
dtype: int64

In [28]:
cp_final_df.groupby("model", dropna=False).size()

model
                            39
BLEND3_3                278043
BLEND_4                  10376
BLEND_REGRESSAO_2026     20654
BVS_CUSTOM                4201
BVS_CUSTOM_V2              162
HFT1                       126
HVA3                     12813
HVA4                      1960
NaN                          2
dtype: int64

In [29]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    327265
E_BVS       1111
dtype: int64

In [30]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)